In [ ]:
import ee
import csv
import json
import os
import zipfile # Thêm thư viện này
from datetime import datetime, timedelta
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

# --- Vùng ĐBSCL ---
MEKONG_PROVINCES = [
    'An Giang','Ben Tre','Ca Mau','Can Tho city','Dong Thap',
    'Hau Giang','Kien Giang','Long An','Soc Trang',
    'Tien Giang','Tra Vinh','Vinh Long','Bac Lieu'
]

# --- Khởi tạo EE ---
try:
    ee.Initialize(project='ee-python-api-471906')
except:
    ee.Authenticate()
    ee.Initialize(project='ee-python-api-471906')

# ---------------- Hàm cơ bản ----------------
def get_province_geometries():
    fcoll = ee.FeatureCollection("FAO/GAUL/2015/level1") \
        .filter(ee.Filter.eq('ADM0_NAME','Viet Nam')) \
        .filter(ee.Filter.inList('ADM1_NAME', MEKONG_PROVINCES))
    features = fcoll.getInfo()['features']
    geoms = {f['properties']['ADM1_NAME']: ee.Geometry(f['geometry']) for f in features}
    return geoms

def get_chirps_daily(region, start_date, end_date):
    return ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY") \
        .filterBounds(region) \
        .filterDate(start_date, end_date)

def get_era5(region, start_date, end_date):
    return ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR") \
        .filterBounds(region) \
        .filterDate(start_date, end_date)

def get_terraclimate(region, start_date, end_date):
    return ee.ImageCollection("IDAHO_EPSCOR/TERRACLIMATE") \
        .filterBounds(region) \
        .filterDate(start_date, end_date)

def get_gldas(region, start_date, end_date):
    return ee.ImageCollection("NASA/GLDAS/V021/NOAH/G025/T3H") \
        .filterBounds(region) \
        .filterDate(start_date, end_date)

# ---------------- Hàm fetch dữ liệu (Không thay đổi) ----------------
# Các hàm fetch_precipitation, fetch_era5, fetch_terraclimate, fetch_gldas không thay đổi

# ... (Giữ nguyên các hàm fetch_precipitation, fetch_era5, fetch_terraclimate, fetch_gldas ở đây) ...

# Hàm fetch_precipitation (chỉ hiển thị phần thay đổi logic thời gian)
def fetch_precipitation(provinces, start_year=2019, end_year=2024):
    """CHIRPS: Lấy tất cả feature (precipitation)"""
    all_data, features_info = [], {"precipitation": {"unit":"mm", "description":"Daily precipitation"}}
    # Thiết lập ngày kết thúc thực tế (cuối 2024)
    end_dt_real = datetime(end_year, 12, 31)

    for prov, geom in provinces.items():
        start_dt = datetime(start_year, 1, 1)
        delta, current_dt = timedelta(days=1), start_dt

        # Chỉ lấy ngày đến hết 31/12/end_year
        total_days = (end_dt_real - start_dt).days + 1

        for _ in tqdm(range(total_days), desc=f"CHIRPS {prov}", leave=True):
            next_dt = current_dt + delta

            # Đảm bảo không vượt quá ngày cuối cùng (end_dt_real)
            if current_dt > end_dt_real:
                break

            # Lấy ảnh
            image = get_chirps_daily(geom, current_dt.strftime("%Y-%m-%d"), next_dt.strftime("%Y-%m-%d")).first()

            if image is not None:
                try:
                    mean_precip = image.reduceRegion(ee.Reducer.mean(), geom, scale=5000, maxPixels=1e9).get('precipitation').getInfo()
                except: mean_precip = None
            else: mean_precip = None

            all_data.append({"province": prov, "date": current_dt.strftime("%Y-%m-%d"), "precipitation": mean_precip})
            current_dt = next_dt

    return all_data, features_info

# Hàm fetch_era5 (chỉ hiển thị phần thay đổi logic thời gian)
def fetch_era5(provinces, start_year=2019, end_year=2024):
    """ERA5: Lấy tất cả các features trung bình hàng ngày"""
    all_data, features_info = [], {}
    # Thiết lập ngày kết thúc thực tế (cuối 2024)
    end_dt_real = datetime(end_year, 12, 31)
    bands = None # Khởi tạo bands ở đây

    for prov, geom in provinces.items():
        start_dt = datetime(start_year, 1, 1)
        delta, current_dt = timedelta(days=1), start_dt
        total_days = (end_dt_real - start_dt).days + 1

        for _ in tqdm(range(total_days), desc=f"ERA5 {prov}", leave=True):
            next_dt = current_dt + delta

            if current_dt > end_dt_real:
                break

            # ERA5 LAND Daily aggregate lấy theo ngày (start_date)
            # Collection.filterDate lấy: [start_date, end_date)
            # Với ảnh daily, nên lấy collection cho cả ngày (current_dt)
            # ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR") chỉ có 1 ảnh/ngày
            collection = get_era5(geom, current_dt.strftime("%Y-%m-%d"), next_dt.strftime("%Y-%m-%d"))
            image = collection.first()

            feature_values = {"province": prov, "date": current_dt.strftime("%Y-%m-%d")}

            if image is not None:
                try:
                    if bands is None:
                        bands = image.bandNames().getInfo()
                        for b in bands:
                            if b not in features_info:
                                # ERA5 bands (tên band)
                                features_info[b] = {"unit": "K", "description": "Thông tin chi tiết về đơn vị (K, m/s, m) có thể tìm thấy trong tài liệu GEE."}
                            # Khởi tạo giá trị None cho tất cả bands
                            feature_values.update({b: None for b in bands})

                    data_dict = image.reduceRegion(ee.Reducer.mean(), geom, scale=10000, maxPixels=1e9).getInfo()
                    if data_dict:
                        feature_values.update(data_dict)
                except Exception:
                    # Nếu có lỗi, giữ nguyên giá trị None đã khởi tạo
                    pass
            else:
                # Nếu không có ảnh, điền None cho tất cả bands nếu đã biết
                if bands:
                    feature_values.update({b: None for b in bands})

            all_data.append(feature_values)
            current_dt = next_dt

    return all_data, features_info

# Hàm fetch_terraclimate (chỉ hiển thị phần thay đổi logic thời gian)
def fetch_terraclimate(provinces, start_year=2019, end_year=2024):
    """TerraClimate: Lấy tất cả các features trung bình tháng"""

    all_data = []
    features_info = {}

    # Định nghĩa trước các band cần chuyển đổi và thông tin đơn vị/mô tả
    BAND_SCALING = {
        'tmmn': {"unit": "°C", "scale": 10, "description": "Nhiệt độ tối thiểu tháng (Tmin). Giá trị thô chia 10."},
        'tmmx': {"unit": "°C", "scale": 10, "description": "Nhiệt độ tối đa tháng (Tmax). Giá trị thô chia 10."},
        'srad': {"unit": "W/m²", "scale": 10, "description": "Bức xạ mặt trời bề mặt (Solar Radiation). Giá trị thô chia 10."},
        'pet': {"unit": "mm", "scale": 1, "description": "Bốc hơi thoát hơi nước tiềm năng (PET)."},
        'aet': {"unit": "mm", "scale": 1, "description": "Bốc hơi thoát hơi nước thực tế (AET)."},
        'pr': {"unit": "mm", "scale": 1, "description": "Lượng mưa (Precipitation)."},
        'ro': {"unit": "mm", "scale": 1, "description": "Dòng chảy (Runoff)."},
        'def': {"unit": "mm", "scale": 1, "description": "Thiếu hụt nước (Climatic Water Deficit)."},
        'soil': {"unit": "mm", "scale": 1, "description": "Độ ẩm đất (Soil Moisture)."},
        'vpd': {"unit": "kPa", "scale": 100, "description": "Áp suất hơi nước bão hòa (Vapor Pressure Deficit). Giá trị thô chia 100."},
        'vap': {"unit": "kPa", "scale": 100, "description": "Áp suất hơi nước (Vapor Pressure). Giá trị thô chia 100."},
        'pdsi': {"unit": "dimensionless", "scale": 100, "description": "Chỉ số hạn hán Palmer (PDSI). Giá trị thô chia 100."},
        'swe': {"unit": "mm", "scale": 1, "description": "Lượng nước tương đương tuyết (Snow Water Equivalent)."},
        'vs': {"unit": "m/s", "scale": 100, "description": "Tốc độ gió (Wind Speed). Giá trị thô chia 100."},
    }

    # Tính tổng số lần lặp cho mỗi tỉnh (2019-2024: 6 năm * 12 tháng)
    num_years = end_year - start_year + 1
    total_months = num_years * 12
    bands = None

    for prov, geom in provinces.items():
        with tqdm(total=total_months, desc=f"TerraClimate {prov}", leave=True) as pbar:
            for year in range(start_year, end_year + 1):
                for month in range(1, 13):
                    # Bỏ qua các tháng sau tháng 12/2024
                    if year > end_year: break

                    start_dt = datetime(year, month, 1)

                    if month == 12:
                        end_dt = datetime(year + 1, 1, 1)
                    else:
                        end_dt = datetime(year, month + 1, 1)

                    collection = get_terraclimate(geom, start_dt.strftime("%Y-%m-%d"), end_dt.strftime("%Y-%m-%d"))
                    image = collection.first()
                    feature_values = {"province": prov, "date": start_dt.strftime("%Y-%m")}

                    if image is not None:
                        try:
                            if bands is None:
                                bands = image.bandNames().getInfo()
                                for b in bands:
                                    if b not in features_info:
                                        info = BAND_SCALING.get(b, {"unit": "Unknown", "scale": 1, "description": "Giá trị trung bình tháng."})
                                        features_info[b] = {"unit": info['unit'], "description": info['description']}

                            data_dict_raw = image.reduceRegion(reducer=ee.Reducer.mean(), geometry=geom, scale=4000, maxPixels=1e9).getInfo()

                            if data_dict_raw:
                                for band_name in bands:
                                    raw_value = data_dict_raw.get(band_name)
                                    scale = BAND_SCALING.get(band_name, {}).get("scale", 1)

                                    if raw_value is not None:
                                        feature_values[band_name] = raw_value / scale
                                    else:
                                        feature_values[band_name] = None
                            else:
                                for b in bands: feature_values[b] = None

                        except Exception:
                            if bands:
                                for b in bands: feature_values[b] = None

                    else:
                        if bands:
                            for b in bands: feature_values[b] = None

                    all_data.append(feature_values)
                    pbar.update(1)

    return all_data, features_info

# Hàm fetch_gldas (chỉ hiển thị phần thay đổi logic thời gian)
def fetch_gldas(provinces, start_year=2019, end_year=2024):
    """GLDAS: Lấy tất cả các features trung bình hàng ngày (Daily Mean)"""

    all_data = []
    features_info = {}

    KNOWN_GLDAS_UNITS = {
        "SoilMoi0_10cm_inst": {"unit": "m3/m3", "description": "Độ ẩm thể tích đất lớp 0-10cm (tức thời)."},
        "Tair_f_inst": {"unit": "K", "description": "Nhiệt độ không khí ở 2m (tức thời)."},
        "Wind_f_inst": {"unit": "m/s", "description": "Vận tốc gió tại 10m (tức thời)."},
        # Thêm các bands khác nếu cần
    }

    # Thiết lập ngày kết thúc thực tế (cuối 2024)
    end_dt_real = datetime(end_year, 12, 31)

    for prov, geom in provinces.items():
        start_dt = datetime(start_year, 1, 1)
        delta, current_dt = timedelta(days=1), start_dt

        total_days = (end_dt_real - start_dt).days + 1
        bands = None

        for _ in tqdm(range(total_days), desc=f"GLDAS {prov}", leave=True):
            next_dt = current_dt + delta
            date_str = current_dt.strftime("%Y-%m-%d")

            if current_dt > end_dt_real:
                break

            collection = get_gldas(geom, date_str, next_dt.strftime("%Y-%m-%d"))
            feature_values = {"province": prov, "date": date_str}

            if collection.size().getInfo() > 0:
                image_mean = collection.mean()

                try:
                    if bands is None:
                        bands = image_mean.bandNames().getInfo()
                        for b in bands:
                            if b not in features_info:
                                info = KNOWN_GLDAS_UNITS.get(b, {"unit": "Unknown", "description": "Giá trị trung bình hàng ngày."})
                                features_info[b] = info
                            feature_values[b] = None

                    data_dict = image_mean.reduceRegion(
                        reducer=ee.Reducer.mean(),
                        geometry=geom,
                        scale=25000,
                        maxPixels=1e9
                    ).getInfo()

                    if data_dict:
                        feature_values.update(data_dict)

                except Exception:
                    pass

            else:
                if bands:
                    for b in bands: feature_values[b] = None

            all_data.append(feature_values)
            current_dt = next_dt

    return all_data, features_info


# ---------------- Hàm Nén và Ghi file CẬP NHẬT ----------------

def zip_folder(folder_path):
    """Nén thư mục thành file zip cùng tên và xóa thư mục gốc."""
    # Tạo tên file zip: data/CHIRPS -> data/CHIRPS.zip
    zip_path = folder_path + ".zip"

    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        # root_dir là thư mục cha (ví dụ: data/)
        root_dir = os.path.dirname(folder_path)
        # base_dir là thư mục cần nén (ví dụ: CHIRPS)
        base_dir = os.path.basename(folder_path)

        for root, dirs, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)
                # Tính toán tên trong file zip (ví dụ: CHIRPS/precipitation_full.csv)
                # Dùng os.path.relpath để loại bỏ folder_path và giữ lại base_dir
                arcname = os.path.join(base_dir, os.path.relpath(file_path, folder_path))
                zipf.write(file_path, arcname)

    print(f"📦 Đã nén thành công: {zip_path}")

    # Xóa thư mục gốc sau khi nén xong
    import shutil
    try:
        shutil.rmtree(folder_path)
        print(f"🗑️ Đã xóa thư mục gốc: {folder_path}")
    except OSError as e:
        print(f"Lỗi khi xóa thư mục {folder_path}: {e}")

def save_and_zip_data(all_data, features_info, folder, name):
    """Ghi dữ liệu ra file, sau đó nén thư mục và xóa thư mục gốc."""
    os.makedirs(folder, exist_ok=True)
    csv_path = os.path.join(folder, f"{name}.csv")
    json_path = os.path.join(folder, f"{name}.json")
    feature_json_path = os.path.join(folder, f"{name}_features_info.json")

    # 1. Ghi file CSV
    if all_data:
        with open(csv_path, "w", newline='', encoding='utf-8') as f:
            # Lấy keys từ entry đầu tiên để làm headers, tránh trường hợp rỗng
            headers = ["province", "date"] + sorted([k for k in all_data[0].keys() if k not in ["province", "date"]])
            writer = csv.DictWriter(f, fieldnames=headers)
            writer.writeheader()
            writer.writerows(all_data)
        print(f"✅ Lưu CSV: {csv_path}")

        # 2. Ghi file JSON
        with open(json_path, "w", encoding='utf-8') as f:
            json.dump(all_data, f, ensure_ascii=False, indent=2)
        print(f"✅ Lưu JSON: {json_path}")

    # 3. Ghi Features info JSON (luôn có)
    with open(feature_json_path, "w", encoding='utf-8') as f:
        json.dump(features_info, f, ensure_ascii=False, indent=2)
    print(f"✅ Lưu Features info JSON: {feature_json_path}")

    # 4. Nén và xóa thư mục
    if os.path.exists(folder) and os.path.isdir(folder):
        zip_folder(folder)


# ---------------- Run Toàn Bộ Dữ Liệu ----------------
if __name__ == "__main__":
    provinces = get_province_geometries()

    # >>> CẤU HÌNH LẠI để chạy toàn bộ 2019 đến 2024
    START_YEAR, END_YEAR = 2019, 2024

    # Tên file đầy đủ
    FULL_NAME = f"mekong_delta_{START_YEAR}_{END_YEAR}_full"

    # ------------------- 1. CHIRPS -------------------
    print("\n\n--- Bắt đầu tải CHIRPS (Daily Precipitation) ---")
    data_chirps, info_chirps = fetch_precipitation(provinces, START_YEAR, END_YEAR)
    save_and_zip_data(data_chirps, info_chirps, "data/CHIRPS", f"chirps_{FULL_NAME}")

    # ------------------- 2. ERA5 -------------------
    print("\n\n--- Bắt đầu tải ERA5-Land (Daily Aggregate) ---")
    data_era5, info_era5 = fetch_era5(provinces, START_YEAR, END_YEAR)
    save_and_zip_data(data_era5, info_era5, "data/ERA5", f"era5_{FULL_NAME}")

    # ------------------- 3. TerraClimate -------------------
    print("\n\n--- Bắt đầu tải TerraClimate (Monthly) ---")
    data_terraclimate, info_terraclimate = fetch_terraclimate(provinces, START_YEAR, END_YEAR)
    save_and_zip_data(data_terraclimate, info_terraclimate, "data/TERRA", f"terraclimate_{FULL_NAME}")

    # ------------------- 4. GLDAS -------------------
    print("\n\n--- Bắt đầu tải GLDAS-Noah (Daily Mean) ---")
    data_gldas, info_gldas = fetch_gldas(provinces, START_YEAR, END_YEAR)
    save_and_zip_data(data_gldas, info_gldas, "data/GLDAS", f"gldas_{FULL_NAME}")

    print("\n\n--- ✅ Hoàn thành tải và nén tất cả dữ liệu! ---")



--- Bắt đầu tải ERA5-Land (Daily Aggregate) ---


ERA5 Tien Giang: 100%|██████████| 2192/2192 [12:10<00:00,  3.00it/s]


✅ Lưu CSV: data/ERA5/era5_mekong_delta_2019_2024_full.csv
✅ Lưu JSON: data/ERA5/era5_mekong_delta_2019_2024_full.json
✅ Lưu Features info JSON: data/ERA5/era5_mekong_delta_2019_2024_full_features_info.json
📦 Đã nén thành công: data/ERA5.zip
🗑️ Đã xóa thư mục gốc: data/ERA5


--- Bắt đầu tải TerraClimate (Monthly) ---


TerraClimate Tien Giang: 100%|██████████| 72/72 [00:20<00:00,  3.52it/s]


✅ Lưu CSV: data/TERRA/terraclimate_mekong_delta_2019_2024_full.csv
✅ Lưu JSON: data/TERRA/terraclimate_mekong_delta_2019_2024_full.json
✅ Lưu Features info JSON: data/TERRA/terraclimate_mekong_delta_2019_2024_full_features_info.json
📦 Đã nén thành công: data/TERRA.zip
🗑️ Đã xóa thư mục gốc: data/TERRA


--- Bắt đầu tải GLDAS-Noah (Daily Mean) ---


GLDAS Tien Giang: 100%|██████████| 2192/2192 [22:53<00:00,  1.60it/s]


✅ Lưu CSV: data/GLDAS/gldas_mekong_delta_2019_2024_full.csv
✅ Lưu JSON: data/GLDAS/gldas_mekong_delta_2019_2024_full.json
✅ Lưu Features info JSON: data/GLDAS/gldas_mekong_delta_2019_2024_full_features_info.json
📦 Đã nén thành công: data/GLDAS.zip
🗑️ Đã xóa thư mục gốc: data/GLDAS


--- ✅ Hoàn thành tải và nén tất cả dữ liệu! ---
